# Full clean headline run — 5 families × 3 seeds × 21 points (Modal, one shot)

Submits all **15 (model × seed) jobs in parallel** into a **fresh run directory** on the volume —
no resume from prior state, nothing deleted, old runs stay archived. Fire-and-forget: once the
submit cell prints its spawn IDs you can close this tab; jobs run to completion server-side.

**Cost, honestly:** ~10–15 A100-hours (~\$25–55) if batching survives the tolerance-1 self-check;
up to ~3× that if models fall back to single-stream. Check one job's log after ~5 min — the
`self-check` line and the first `gen/s | ETA` heartbeat tell you which regime you're in.

In [ ]:
!pip install -q modal
# Paste your (rotated!) Modal token line:
!modal token set --token-id ak-XXXX --token-secret as-XXXX --profile=willgray
!modal profile activate willgray

In [ ]:
import datetime, os
RUN_DIR = 'run-' + datetime.datetime.utcnow().strftime('%Y%m%d-%H%M')
os.environ['RUN_DIR'] = RUN_DIR
print('this run writes to  necessity-results:/' + RUN_DIR)

In [ ]:
# Deploy current pipeline code, then spawn all 15 jobs (fire-and-forget)
!rm -rf /content/repo && git clone -q https://github.com/wrgr/socratic-scenarios /content/repo
!modal deploy /content/repo/experiments/modal_headline.py
!python /content/repo/experiments/modal_submit.py

## Status — run any time (safe after reconnecting; re-run the auth + RUN_DIR cells first)

In [ ]:
!modal volume ls necessity-results /$RUN_DIR 2>/dev/null | grep -c jsonl || echo '0 (dir not created yet)'
print('want 315 transcripts (15 jobs x 21 alphas); each job banks its files as it finishes')

## Pull when green — zips just this run and parks it in Drive

In [ ]:
!mkdir -p /content/pull && modal volume get necessity-results /$RUN_DIR /content/pull --force
!find /content/pull -name 'dose_factqa_*_a*.jsonl' | wc -l
!cd /content && zip -qr factqa_$RUN_DIR.zip pull
from google.colab import drive; drive.mount('/content/drive')
import os, shutil
os.makedirs('/content/drive/MyDrive/necessity-audit', exist_ok=True)
shutil.copy(f"/content/factqa_{os.environ['RUN_DIR']}.zip", '/content/drive/MyDrive/necessity-audit/')
print('zip in Drive/necessity-audit — upload it to the session for scoring + paper update')